In [ ]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from IPython.display import HTML

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
from google.colab import files
files.upload()

Saving model_8.pth to model_8.pth
Saving model_4.pth to model_4.pth
Saving sprites_1788_16x16.npy to sprites_1788_16x16.npy
Saving sprite_labels_nc_1788_16x16.npy to sprite_labels_nc_1788_16x16.npy
Saving model_20.pth to model_20.pth
Saving model_31.pth to model_31.pth
Saving diffusion_utilities.py to diffusion_utilities.py


In [ ]:
from diffusion_utilities import (
    ResidualConvBlock,
    UnetUp,
    UnetDown,
    EmbedFC,
    plot_sample,
    CustomDataset,
    transform
)

In [ ]:
# diffusion hyperparameters
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

# network hyperparameters
n_feat = 64
n_cfeat = 5
height = 16

# training hyperparameters
batch_size = 100
n_epoch = 32
lrate = 1e-3

save_dir = "./weights/"

In [ ]:
b_t = beta1 + (beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device)
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1

In [ ]:
class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat=256, n_cfeat=10, height=28):
        super(ContextUnet, self).__init__()

        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_cfeat = n_cfeat
        self.h = height

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)

        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)

        self.to_vec = nn.Sequential(nn.AvgPool2d((4)), nn.GELU())

        self.timeembed1 = EmbedFC(1, 2 * n_feat)
        self.timeembed2 = EmbedFC(1, 1 * n_feat)
        self.contextembed1 = EmbedFC(n_cfeat, 2 * n_feat)
        self.contextembed2 = EmbedFC(n_cfeat, 1 * n_feat)

        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, self.h // 4, self.h // 4),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )

        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)

        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, t, c=None):
        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)

        hiddenvec = self.to_vec(down2)

        if c is None:
            c = torch.zeros(x.shape[0], self.n_cfeat).to(x)

        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1 * up1 + temb1, down2)
        up3 = self.up2(cemb2 * up2 + temb2, down1)

        out = self.out(torch.cat((up3, x), 1))
        return out

In [ ]:
nn_model = ContextUnet(in_channels=3, n_feat=n_feat, n_cfeat=n_cfeat, height=height).to(device)
print("Model initialized")

In [ ]:
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise

In [ ]:
@torch.no_grad()
def sample_ddpm(n_sample, save_rate=20):
    samples = torch.randn(n_sample, 3, height, height).to(device)

    intermediate = []
    for i in range(timesteps, 0, -1):
        print(f'sampling timestep {i:3d}', end='\r')

        t = torch.tensor([i / timesteps])[:, None, None, None].to(device)
        z = torch.randn_like(samples) if i > 1 else 0

        eps = nn_model(samples, t)
        samples = denoise_add_noise(samples, i, eps, z)

        if i % save_rate == 0 or i == timesteps or i < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate

In [ ]:
dataset = CustomDataset(
    "./sprites_1788_16x16.npy",
    "./sprite_labels_nc_1788_16x16.npy",
    transform,
    null_context=False
)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=1)
optim = torch.optim.Adam(nn_model.parameters(), lr=lrate)

In [ ]:
def plot_sample_images(loader):
    images, _ = next(iter(loader))
    plt.figure(figsize=(10, 5))
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].permute(1, 2, 0).cpu().numpy())
        plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_sample_images(dataloader)

In [ ]:
def perturb_input(x, t, noise):
    return ab_t.sqrt()[t, None, None, None] * x + (1 - ab_t[t, None, None, None]).sqrt() * noise

In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""
### Question

Can you explain the components: $x_t$, $x_0$, $\bar{\alpha}_t$, $\epsilon$ ?

### Réponse

- **$x_t$** : image bruitée à l’étape $t$.
- **$x_0$** : image originale propre, avant ajout de bruit.
- **$\bar{\alpha}_t$** : produit cumulé des coefficients de conservation du signal jusqu’au temps $t$, qui indique la quantité d’information de l’image initiale encore présente.
- **$\epsilon$** : bruit gaussien aléatoire ajouté à l’image pendant le processus de diffusion.
"""))

In [ ]:
nn_model.train()

for ep in range(n_epoch):
    print(f'epoch {ep}')

    optim.param_groups[0]['lr'] = lrate * (1 - ep / n_epoch)

    pbar = tqdm(dataloader, mininterval=2)
    for x, _ in pbar:
        optim.zero_grad()
        x = x.to(device)

        noise = torch.randn_like(x)
        t = torch.randint(1, timesteps + 1, (x.shape[0],), device=device)

        x_pert = perturb_input(x, t, noise)

        pred_noise = nn_model(x_pert, t / timesteps)

        loss = F.mse_loss(pred_noise, noise)
        loss.backward()

        optim.step()

        pbar.set_description(f"loss: {loss.item():.4f}")

    if ep % 4 == 0 or ep == int(n_epoch - 1):
        if not os.path.exists(save_dir):
            os.mkdir(save_dir)
        torch.save(nn_model.state_dict(), save_dir + f"model_{ep}.pth")
        print('saved model at ' + save_dir + f"model_{ep}.pth")

epoch 0


loss: 0.1366: 100%|██████████| 894/894 [00:39<00:00, 22.73it/s]


saved model at ./weights/model_0.pth
epoch 1


loss: 0.1209: 100%|██████████| 894/894 [00:31<00:00, 28.10it/s]


epoch 2


...

epoch 30


loss: 0.0455: 100%|██████████| 894/894 [00:32<00:00, 27.40it/s]


epoch 31


loss: 0.0331: 100%|██████████| 894/894 [00:32<00:00, 27.17it/s]

saved model at ./weights/model_31.pth


In [ ]:
nn_model.load_state_dict(torch.load(f"{save_dir}/model_0.pth", map_location=device))
nn_model.eval()
print("Loaded in Model")

Loaded in Model


In [ ]:
plt.clf()

samples, intermediate_ddpm = sample_ddpm(32)

animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, save_dir, "ani_run", None, save=False)

HTML(animation_ddpm.to_jshtml())

<Figure size 640x480 with 0 Axes>

In [ ]:
def evaluate_checkpoint(epoch):
    nn_model.load_state_dict(torch.load(f"./model_{epoch}.pth", map_location=device))
    nn_model.eval()
    print(f"Loaded model_{epoch}.pth")

    plt.clf()
    samples, intermediate_ddpm = sample_ddpm(32)
    animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, "", f"ani_run_{epoch}", None, save=False)
    return HTML(animation_ddpm.to_jshtml())

In [ ]:
evaluate_checkpoint(4)

In [ ]:
evaluate_checkpoint(8)

Loaded model_8.pth


<Figure size 640x480 with 0 Axes>

In [ ]:
evaluate_checkpoint(20)

In [ ]:
evaluate_checkpoint(31)

In [ ]:
display(Markdown(r"""
### Observation

Quand le nombre d’époques augmente, la qualité des images générées s’améliore progressivement.

- **Epoch 4** : images encore très bruitées, formes peu reconnaissables.
- **Epoch 8** : les structures commencent à apparaître, mais restent floues.
- **Epoch 20** : les sprites sont plus cohérents et mieux définis.
- **Epoch 31** : les images sont globalement plus nettes, plus réalistes et plus stables.

On observe donc que le modèle apprend progressivement à prédire et retirer le bruit de manière plus précise au fil de l’entraînement.
"""))

In [ ]:
def denoise_ddim(x, t, t_prev, pred_noise):
    ab = ab_t[t]
    ab_prev = ab_t[t_prev]

    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
    dir_xt = (1 - ab_prev).sqrt() * pred_noise

    return ab_prev.sqrt() * x0_pred + dir_xt

In [ ]:
@torch.no_grad()
def sample_ddim(n_sample, n=20):
    samples = torch.randn(n_sample, 3, height, height).to(device)

    intermediate = []
    step_size = timesteps // n

    for i in range(timesteps, 0, -step_size):
        print(f'sampling timestep {i:3d}', end='\r')

        t = torch.tensor([i / timesteps])[:, None, None, None].to(device)

        eps = nn_model(samples, t)
        t_prev = max(i - step_size, 1)
        samples = denoise_ddim(samples, i, t_prev, eps)

        intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate

In [ ]:
plt.clf()

samples, intermediate = sample_ddim(32, n=25)

animation_ddim = plot_sample(intermediate, 32, 4, save_dir, "ani_run_ddim", None, save=False)

HTML(animation_ddim.to_jshtml())

In [ ]:
display(Markdown(r"""
### Remarque

Le **DDPM** génère les images avec beaucoup d’étapes et un processus stochastique, alors que le **DDIM** utilise un processus plus déterministe et plus rapide.

Le DDIM permet donc d’obtenir des résultats visuellement corrects avec moins d’étapes de sampling.
"""))


### Remarque

Le **DDPM** génère les images avec beaucoup d’étapes et un processus stochastique, alors que le **DDIM** utilise un processus plus déterministe et plus rapide.

Le DDIM permet donc d’obtenir des résultats visuellement corrects avec moins d’étapes de sampling.
